In [1]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
from pathlib import Path
import os

In [2]:
configs = {
    "M2Hex": {
        "block_number": "T349",
        "program": "BLOCK-T349",
        "perturbations": {
            "M2_dz": 100.0,
            "M2_dx": 1000.0,
            "M2_dy": 1000.0,
            "M2_rx": 0.06, # degrees
            "M2_ry": 0.06, # degrees
        },
    },
    "CamHex": {
        "block_number": "T350",
        "program": "BLOCK-T350",
        "perturbations": {
            "Cam_dz": 100.0,
            "Cam_dx": 1000.0,
            "Cam_dy": 1000.0,
            "Cam_rx": 0.06, # degrees
            "Cam_ry": 0.06, # degrees
        },
    },
    "M1M3Bend": {
        "block_number": "T351",
        "program": "BLOCK-T351",
        "perturbations": {
            "M1M3_B1": 3.0,
            "M1M3_B2": 3.0,
            "M1M3_B3": 3.0,
            "M1M3_B4": 3.0,
            "M1M3_B5": 3.0,
            "M1M3_B6": 0.5,
            "M1M3_B7": 0.5,
            "M1M3_B8": 0.5,
            "M1M3_B9": 0.5,
            "M1M3_B10": 0.5,
            "M1M3_B11": 0.5,    
            "M1M3_B12": 0.5,
            "M1M3_B13": 0.4,
            "M1M3_B14": 0.4,
            "M1M3_B15": 0.2,
            "M1M3_B16": 0.2,
            "M1M3_B17": 0.4,
            "M1M3_B18": 0.4,
            "M1M3_B19": 0.3,
            "M1M3_B20": 0.1,
        },
    },
    "M2Bend": {
        "block_number": "T352",
        "program": "BLOCK-T352",
        "perturbations": {
            "M2_B1": 3.0,
            "M2_B2": 3.0,
            "M2_B3": 3.0,
            "M2_B4": 3.0,
            "M2_B5": 3.0,
            "M2_B6": 0.5,
            "M2_B7": 0.5,
            "M2_B8": 1.0,
            "M2_B9": 1.0,
            "M2_B10": 0.5,
            "M2_B11": 0.5,
            "M2_B12": 0.5,
            "M2_B13": 0.5,
            "M2_B14": 0.4,
            "M2_B15": 0.4,
            "M2_B16": 0.3,
            "M2_B17": 0.3,
            "M2_B18": 0.1,
            "M2_B19": 0.1,
            "M2_B20": 0.1,
        },
    },
}

In [4]:
for name, config in configs.items():
    properties = {
        "run_filter": {
            "description": "ComCam filter to use.",
            "type": "string",
            "default": "r_03"
        },
        "exposure_time": {
            "description": "Exposure time in seconds.",
            "type": "number",
            "default": 0.2
        },
        "n_test_images": {
            "description": "Number of test images for each dof.",
            "type": "number",
            "default": 1
        }
    }
    configuration_schema = build_configuration_schema(
        config['block_number'], properties
    )    

    scripts = []

    # Start with 3 baseline exposures
    scripts.append(
        ObservingScript(
            name="maintel/take_image_comcam.py",
            standard=True,
            parameters=dict(
                filter="$run_filter",
                program="$program",
                reason="spot_dance_baseline",
                exp_times="$exposure_time",
                image_type="ENGTEST",
                nimages=3            
            )
        )
    )

    # Loop through each dof, setting current perturbation and unsetting previous

    prev = None
    for k, v in config['perturbations'].items():
        apply_dof_parameters = dict()
        if prev is not None:  # undo previous perturbation
            apply_dof_parameters[prev[0]] = -prev[1]
        apply_dof_parameters[k]=v
        scripts.append(
            ObservingScript(
                name="maintel/apply_dof.py",
                standard=True,
                parameters=apply_dof_parameters
            )
        )
        scripts.append(
            ObservingScript(
                name="maintel/take_image_comcam.py",
                standard=True,
                parameters=dict(
                    filter="$run_filter",
                    program="$program",
                    reason="spot_dance_test",
                    note=f"{k}={v}",
                    exp_times="$exposure_time",
                    image_type="ENGTEST",
                    nimages="$n_test_images"
                )
            )
        )
        prev = k, v
    
    # reset last perturbation
    apply_dof_parameters = dict()
    apply_dof_parameters[prev[0]] = -prev[1]
    scripts.append(
        ObservingScript(
            name="maintel/apply_dof.py",
            standard=True,
            parameters=apply_dof_parameters
        )
    )

    # Final baseline
    scripts.append(
        ObservingScript(
            name="maintel/take_image_comcam.py",
            standard=True,
            parameters=dict(
                filter="$run_filter",
                program="$program",
                reason="spot_dance_baseline",
                exp_times="$exposure_time",
                image_type="ENGTEST",
                nimages=3
            )
        )
    )

    block = ObservingBlock(
        name = config['program'],
        program = config['program'],
        configuration_schema=configuration_schema,
        scripts = scripts,
    )

    # print(block.model_dump_json(indent=2))

    ocs_path = Path(os.environ['TS_CONFIG_OCS_DIR'])
    file_path = ocs_path / "Scheduler" / "observing_blocks_maintel" / "AOS" / "CBP"
    file_path /= f"{config['program']}.json"
    
    with open(file_path, 'w') as f:
        f.write(block.model_dump_json(indent=2))